In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/trueestate_3cities_accessibility.csv"
)

print("=" * 60)
print("FINAL TRAINING DATASET")
print("=" * 60)

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nCities:")
print(df["city"].value_counts())

print("\nTarget:")
print(df["rent"].describe())

FINAL TRAINING DATASET
Shape: (5286, 5491)
Missing values: 848
Duplicate rows: 0

Cities:
city
New Delhi    1799
Bangalore    1786
Mumbai       1701
Name: count, dtype: int64

Target:
count    5.286000e+03
mean     6.357727e+04
std      9.899721e+04
min      1.000000e+03
25%      1.800000e+04
50%      3.500000e+04
75%      7.000000e+04
max      2.700000e+06
Name: rent, dtype: float64


In [3]:
missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

print("=" * 70)
print("COLUMNS WITH MISSING VALUES")
print("=" * 70)

print("Number of columns with missing values:", len(missing))
print("\nMissing values by column:")
print(missing)

print("\nTotal missing:", missing.sum())

COLUMNS WITH MISSING VALUES
Number of columns with missing values: 2

Missing values by column:
latitude     424
longitude    424
dtype: int64

Total missing: 848


In [4]:
important_cols = [
    "hospital_km",
    "school_km",
    "mall_km",
    "station_km",
    "accessibility_score",
    "accessibility_available",
    "rent"
]

print("\nIMPORTANT FEATURE CHECK")
print("=" * 70)

for col in important_cols:
    if col in df.columns:
        print(f"{col:25} missing = {df[col].isna().sum()}")
    else:
        print(f"{col:25} ❌ COLUMN NOT FOUND")


IMPORTANT FEATURE CHECK
hospital_km               missing = 0
school_km                 missing = 0
mall_km                   missing = 0
station_km                missing = 0
accessibility_score       missing = 0
accessibility_available   missing = 0
rent                      missing = 0


In [5]:
print("=" * 70)
print("COLUMN AUDIT")
print("=" * 70)

# Object/string columns
object_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Object columns:")
print(object_cols)

# Potential leakage / non-model columns
check_cols = [
    "rent",
    "area_rate",
    "latitude",
    "longitude",
    "locality",
    "city"
]

print("\nImportant columns:")

for col in check_cols:
    print(
        f"{col:15}",
        "FOUND" if col in df.columns else "NOT FOUND"
    )

print("\nTotal numeric columns:",
      len(df.select_dtypes(include=np.number).columns))

print("Total object columns:", len(object_cols))

COLUMN AUDIT
Object columns:
['locality', 'city']

Important columns:
rent            FOUND
area_rate       FOUND
latitude        FOUND
longitude       FOUND
locality        FOUND
city            FOUND

Total numeric columns: 17
Total object columns: 2


C:\Users\kopal\AppData\Local\Temp\ipykernel_7536\3312400879.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [6]:
print("=" * 70)
print("AREA RATE CHECK")
print("=" * 70)

print(df[["rent", "area_rate", "area"]].corr())

print("\nArea rate summary:")
print(df["area_rate"].describe())

AREA RATE CHECK
               rent  area_rate      area
rent       1.000000   0.617028  0.440201
area_rate  0.617028   1.000000  0.023606
area       0.440201   0.023606  1.000000

Area rate summary:
count    5286.000000
mean       53.345882
std        44.623803
min         2.666667
25%        24.000000
50%        37.000000
75%        70.000000
max       300.000000
Name: area_rate, dtype: float64


In [7]:
# ============================================================
# CHECK WHETHER AREA_RATE IS DERIVED FROM RENT
# ============================================================

calculated_area_rate = df["rent"] / df["area"]

comparison = pd.DataFrame({
    "stored_area_rate": df["area_rate"],
    "rent_div_area": calculated_area_rate
})

comparison["difference"] = (
    comparison["stored_area_rate"] -
    comparison["rent_div_area"]
).abs()

print("=" * 70)
print("AREA RATE LEAKAGE TEST")
print("=" * 70)

print("\nMean absolute difference:")
print(comparison["difference"].mean())

print("\nMedian absolute difference:")
print(comparison["difference"].median())

print("\nPercentage within 1 unit:")
print(
    (comparison["difference"] <= 1).mean() * 100
)

print("\nPercentage within 5 units:")
print(
    (comparison["difference"] <= 5).mean() * 100
)

print("\nSample:")
print(comparison.head(15))

AREA RATE LEAKAGE TEST

Mean absolute difference:
18.215433120046747

Median absolute difference:
0.23913833412573737

Percentage within 1 unit:
94.66515323496027

Percentage within 5 units:
98.69466515323496

Sample:
    stored_area_rate  rent_div_area  difference
0         134.000000     133.779264    0.220736
1          82.000000      81.632653    0.367347
2          25.000000      25.210084    0.210084
3          27.000000      26.666667    0.333333
4          47.666667      47.619048    0.047619
5          74.111111      74.074074    0.037037
6          40.000000      40.000000    0.000000
7          39.000000      38.885288    0.114712
8         184.000000     184.210526    0.210526
9         222.000000     226.385636    4.385636
10        159.000000     159.090909    0.090909
11         92.000000      92.307692    0.307692
12         23.000000      23.148148    0.148148
13         19.000000      18.571429    0.428571
14         37.000000      36.842105    0.157895


In [8]:
from sklearn.model_selection import train_test_split

# Keep the full dataframe for now
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining cities:")
print(train_df["city"].value_counts())

print("\nTesting cities:")
print(test_df["city"].value_counts())

TRAIN / TEST SPLIT
Training rows: 4228
Testing rows: 1058

Training cities:
city
Bangalore    1434
New Delhi    1434
Mumbai       1360
Name: count, dtype: int64

Testing cities:
city
New Delhi    365
Bangalore    352
Mumbai       341
Name: count, dtype: int64


In [9]:
# ============================================================
# LOCALITY MARKET RATE - TRAINING DATA ONLY
# ============================================================

# area_rate itself is rent / area, so calculate the historical
# locality statistic ONLY from training rows.

locality_market_rates = (
    train_df
    .groupby(["city", "locality"])["area_rate"]
    .median()
)

city_market_rates = (
    train_df
    .groupby("city")["area_rate"]
    .median()
)

print("Locality market rates created:", len(locality_market_rates))

print("\nCity fallback rates:")
print(city_market_rates)

Locality market rates created: 1330

City fallback rates:
city
Bangalore    30.0
Mumbai       82.5
New Delhi    28.0
Name: area_rate, dtype: float64


In [10]:
def add_market_rate(data):

    data = data.copy()

    keys = pd.MultiIndex.from_frame(
        data[["city", "locality"]]
    )

    data["locality_market_rate"] = (
        locality_market_rates.reindex(keys).to_numpy()
    )

    # Unseen locality → use training-set city median
    data["locality_market_rate"] = (
        data["locality_market_rate"]
        .fillna(data["city"].map(city_market_rates))
    )

    return data


train_df = add_market_rate(train_df)
test_df = add_market_rate(test_df)

print("=" * 70)
print("MARKET RATE FEATURE")
print("=" * 70)

print(
    "Train missing:",
    train_df["locality_market_rate"].isna().sum()
)

print(
    "Test missing:",
    test_df["locality_market_rate"].isna().sum()
)

print("\nTraining market rate:")
print(train_df["locality_market_rate"].describe())

print("\nTesting market rate:")
print(test_df["locality_market_rate"].describe())

MARKET RATE FEATURE
Train missing: 0
Test missing: 0

Training market rate:
count    4228.000000
mean       51.272312
std        38.406436
min         4.666667
25%        25.000000
50%        36.000000
75%        72.000000
max       300.000000
Name: locality_market_rate, dtype: float64

Testing market rate:
count    1058.000000
mean       51.809651
std        38.432094
min         6.000000
25%        28.000000
50%        35.000000
75%        73.000000
max       230.000000
Name: locality_market_rate, dtype: float64


In [11]:
exclude_cols = [
    "rent",          # target
    "area_rate",     # leakage
    "latitude",      # raw coordinate
    "longitude",     # raw coordinate
    "locality",      # raw text
    "city"           # raw text
]

X_train = train_df.drop(columns=exclude_cols)
X_test  = test_df.drop(columns=exclude_cols)

y_train = train_df["rent"].copy()
y_test  = test_df["rent"].copy()

print("=" * 70)
print("FINAL MODEL MATRICES")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nMissing X_train:", X_train.isna().sum().sum())
print("Missing X_test :", X_test.isna().sum().sum())

print("\nNon-numeric columns:")
print(X_train.select_dtypes(exclude=np.number).columns.tolist())

print("\nLeakage check:")
for col in ["rent", "area_rate"]:
    print(
        col,
        "❌ FOUND" if col in X_train.columns else "✅ EXCLUDED"
    )

print("\nNew features:")
for col in [
    "locality_market_rate",
    "hospital_km",
    "school_km",
    "mall_km",
    "station_km",
    "accessibility_score",
    "accessibility_available"
]:
    print(
        f"{col:27}",
        "✅" if col in X_train.columns else "❌"
    )

FINAL MODEL MATRICES
X_train: (4228, 5486)
X_test : (1058, 5486)
y_train: (4228,)
y_test : (1058,)

Missing X_train: 0
Missing X_test : 0

Non-numeric columns:
['house_type_1 BHK Flat for Rent in 7th Heaven, Dhanori, Pune', 'house_type_1 BHK Flat for Rent in Aaditya Glory II, Godhani, Nagpur', 'house_type_1 BHK Flat for Rent in Abhimaan Homes, Shirgaon, Pune', 'house_type_1 BHK Flat for Rent in Abhyudaya Nagar, Mumbai', 'house_type_1 BHK Flat for Rent in Acme Aakansha I, Goregaon Mulund Link Road, Mumbai', 'house_type_1 BHK Flat for Rent in Acme Harmony Chs Ltd, Andheri East, Mumbai', 'house_type_1 BHK Flat for Rent in Adarsh Nagar Lohegaon, Pune', 'house_type_1 BHK Flat for Rent in Adarsh Nagar, New Delhi', 'house_type_1 BHK Flat for Rent in Adithansh Apartment, Kondhwa, Pune', 'house_type_1 BHK Flat for Rent in Aditya Shagun, Bavdhan, Pune', 'house_type_1 BHK Flat for Rent in Aga Nagar, Vadgaonsheri, Pune', 'house_type_1 BHK Flat for Rent in Aishwarya Aangan, Chakan, Pune', 'house_ty

In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# ============================================================
# TRAIN FINAL XGBOOST MODEL
# ============================================================

model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training model...")

model.fit(
    X_train,
    y_train
)

print("✅ Training complete")

Training model...
✅ Training complete


In [13]:
# ============================================================
# EVALUATION
# ============================================================

pred = model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("=" * 70)
print("FINAL XGBOOST RESULTS")
print("=" * 70)

print(f"R²   : {r2:.4f}")
print(f"MAE  : ₹{mae:,.2f}")
print(f"RMSE : ₹{rmse:,.2f}")

FINAL XGBOOST RESULTS
R²   : 0.6929
MAE  : ₹21,696.88
RMSE : ₹53,249.93


In [14]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred
})

results["absolute_error"] = (
    results["actual"] - results["predicted"]
).abs()

results["rent_band"] = pd.cut(
    results["actual"],
    bins=[0, 20000, 50000, 100000, np.inf],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ]
)

print("\nMAE BY RENT BAND")
print("=" * 70)

print(
    results.groupby(
        "rent_band",
        observed=True
    )["absolute_error"]
    .agg(["count", "mean", "median"])
    .round(2)
)


MAE BY RENT BAND
           count      mean    median
rent_band                           
<₹20k        332   7061.41   4441.34
₹20k–₹50k    349  11221.70   7645.06
₹50k–₹1L     219  21103.96  14175.10
>₹1L         158  76409.96  44902.07


In [15]:
# ============================================================
# LOG-TARGET XGBOOST
# ============================================================

y_train_log = np.log1p(y_train)

log_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.025,
    max_depth=6,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training log-target model...")

log_model.fit(
    X_train,
    y_train_log
)

# Convert predictions back to ₹
pred_log = np.expm1(
    log_model.predict(X_test)
)

pred_log = np.maximum(pred_log, 0)

print("✅ Training complete")

Training log-target model...
✅ Training complete


In [16]:
r2_log = r2_score(y_test, pred_log)
mae_log = mean_absolute_error(y_test, pred_log)
rmse_log = np.sqrt(
    mean_squared_error(y_test, pred_log)
)

print("=" * 70)
print("LOG-TARGET XGBOOST RESULTS")
print("=" * 70)

print(f"R²   : {r2_log:.4f}")
print(f"MAE  : ₹{mae_log:,.2f}")
print(f"RMSE : ₹{rmse_log:,.2f}")

LOG-TARGET XGBOOST RESULTS
R²   : 0.6787
MAE  : ₹21,130.21
RMSE : ₹54,466.45


In [17]:
log_results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred_log
})

log_results["absolute_error"] = (
    log_results["actual"] -
    log_results["predicted"]
).abs()

log_results["rent_band"] = pd.cut(
    log_results["actual"],
    bins=[0, 20000, 50000, 100000, np.inf],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ]
)

print("LOG MODEL — MAE BY RENT BAND")
print("=" * 70)

print(
    log_results.groupby(
        "rent_band",
        observed=True
    )["absolute_error"]
    .agg(["count", "mean", "median"])
    .round(2)
)

LOG MODEL — MAE BY RENT BAND
           count      mean    median
rent_band                           
<₹20k        332   6359.48   4418.87
₹20k–₹50k    349  10136.91   7196.81
₹50k–₹1L     219  19724.31  15075.86
>₹1L         158  78398.81  51322.50


In [18]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

# Remove the previous training market rate
train_df = train_df.drop(
    columns=["locality_market_rate"],
    errors="ignore"
)

test_df = test_df.drop(
    columns=["locality_market_rate"],
    errors="ignore"
)

# ============================================================
# OUT-OF-FOLD LOCALITY MARKET RATE
# ============================================================

train_df["locality_market_rate"] = np.nan

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (fit_idx, val_idx) in enumerate(kf.split(train_df), 1):

    fit_part = train_df.iloc[fit_idx]
    val_part = train_df.iloc[val_idx]

    # Rates learned ONLY from the other folds
    locality_rates = (
        fit_part
        .groupby(["city", "locality"])["area_rate"]
        .median()
    )

    city_rates = (
        fit_part
        .groupby("city")["area_rate"]
        .median()
    )

    keys = pd.MultiIndex.from_frame(
        val_part[["city", "locality"]]
    )

    mapped = locality_rates.reindex(keys).to_numpy()

    mapped = pd.Series(
        mapped,
        index=val_idx
    )

    # Unseen locality -> city median
    city_fallback = (
        val_part["city"]
        .map(city_rates)
        .to_numpy()
    )

    mapped = mapped.fillna(
        pd.Series(
            city_fallback,
            index=val_idx
        )
    )

    train_df.iloc[
        val_idx,
        train_df.columns.get_loc("locality_market_rate")
    ] = mapped.to_numpy()

    print(f"Fold {fold} complete")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [19]:
# ============================================================
# TEST MARKET RATE FROM TRAINING DATA ONLY
# ============================================================

final_locality_rates = (
    train_df
    .groupby(["city", "locality"])["area_rate"]
    .median()
)

final_city_rates = (
    train_df
    .groupby("city")["area_rate"]
    .median()
)

test_keys = pd.MultiIndex.from_frame(
    test_df[["city", "locality"]]
)

test_df["locality_market_rate"] = (
    final_locality_rates
    .reindex(test_keys)
    .to_numpy()
)

test_df["locality_market_rate"] = (
    test_df["locality_market_rate"]
    .fillna(
        test_df["city"].map(final_city_rates)
    )
)

print("=" * 70)
print("OOF MARKET RATE CHECK")
print("=" * 70)

print(
    "Train missing:",
    train_df["locality_market_rate"].isna().sum()
)

print(
    "Test missing:",
    test_df["locality_market_rate"].isna().sum()
)

print("\nTrain:")
print(train_df["locality_market_rate"].describe())

print("\nTest:")
print(test_df["locality_market_rate"].describe())

OOF MARKET RATE CHECK
Train missing: 0
Test missing: 0

Train:
count    4228.000000
mean       51.060641
std        36.585728
min         7.000000
25%        28.000000
50%        34.000000
75%        73.500000
max       275.000000
Name: locality_market_rate, dtype: float64

Test:
count    1058.000000
mean       51.809651
std        38.432094
min         6.000000
25%        28.000000
50%        35.000000
75%        73.000000
max       230.000000
Name: locality_market_rate, dtype: float64


In [20]:
exclude_cols = [
    "rent",
    "area_rate",
    "latitude",
    "longitude",
    "locality",
    "city"
]

X_train = train_df.drop(columns=exclude_cols)
X_test  = test_df.drop(columns=exclude_cols)

y_train = train_df["rent"].copy()
y_test  = test_df["rent"].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Missing train:", X_train.isna().sum().sum())
print("Missing test :", X_test.isna().sum().sum())

X_train: (4228, 5486)
X_test : (1058, 5486)
Missing train: 0
Missing test : 0


In [21]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

model_oof = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.025,
    max_depth=6,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_oof.fit(X_train, y_train)

pred_oof = model_oof.predict(X_test)

r2_oof = r2_score(y_test, pred_oof)
mae_oof = mean_absolute_error(y_test, pred_oof)
rmse_oof = np.sqrt(mean_squared_error(y_test, pred_oof))

print("=" * 70)
print("OOF MARKET-RATE XGBOOST")
print("=" * 70)
print(f"R²   : {r2_oof:.4f}")
print(f"MAE  : ₹{mae_oof:,.2f}")
print(f"RMSE : ₹{rmse_oof:,.2f}")

OOF MARKET-RATE XGBOOST
R²   : 0.6701
MAE  : ₹22,026.88
RMSE : ₹55,189.37


In [22]:
oof_results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred_oof
})

oof_results["absolute_error"] = (
    oof_results["actual"] - oof_results["predicted"]
).abs()

oof_results["rent_band"] = pd.cut(
    oof_results["actual"],
    bins=[0, 20000, 50000, 100000, np.inf],
    labels=["<₹20k", "₹20k–₹50k", "₹50k–₹1L", ">₹1L"]
)

print(
    oof_results.groupby(
        "rent_band",
        observed=True
    )["absolute_error"]
    .agg(["count", "mean", "median"])
    .round(2)
)

           count      mean    median
rent_band                           
<₹20k        332   6910.60   3991.10
₹20k–₹50k    349  11986.76   7621.16
₹50k–₹1L     219  21762.11  13553.89
>₹1L         158  76334.43  43486.17


In [23]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_oof.feature_importances_
}).sort_values("importance", ascending=False)

print("=" * 70)
print("TOP 30 FEATURES")
print("=" * 70)

print(
    importance.head(30).to_string(index=False)
)

TOP 30 FEATURES
                                                                          feature  importance
                                                                        bathrooms    0.301766
                                                             locality_market_rate    0.072654
                                                                      city_Mumbai    0.054319
                                                                             area    0.050670
                            house_type_4 BHK Flat for Rent in Bandra West, Mumbai    0.045186
                                                                             beds    0.042492
                                                                 area_per_bedroom    0.024911
       house_type_3 BHK Flat for Rent in Lokhandwala Minerva, Mahalakshmi, Mumbai    0.019995
                                                                        balconies    0.015556
            house_type_5 BHK Flat for Rent i

In [24]:
new_features = [
    "locality_market_rate",
    "hospital_km",
    "school_km",
    "mall_km",
    "station_km",
    "accessibility_score",
    "accessibility_available"
]

print("\n" + "=" * 70)
print("NEW FEATURE IMPORTANCE")
print("=" * 70)

print(
    importance[
        importance["feature"].isin(new_features)
    ].to_string(index=False)
)


NEW FEATURE IMPORTANCE
                feature  importance
   locality_market_rate    0.072654
    accessibility_score    0.013551
            hospital_km    0.013329
             station_km    0.012390
                mall_km    0.010482
              school_km    0.008268
accessibility_available    0.007527


In [25]:
print("=" * 70)
print("HOUSE TYPE ANALYSIS")
print("=" * 70)

house_cols = [
    col for col in df.columns
    if col.startswith("house_type_")
]

print("Number of house_type columns:", len(house_cols))

print("\nSample:")
for col in house_cols[:30]:
    print(col)

HOUSE TYPE ANALYSIS
Number of house_type columns: 5466

Sample:
house_type_1 BHK Flat for Rent in 7th Heaven, Dhanori, Pune
house_type_1 BHK Flat for Rent in Aaditya Glory II, Godhani, Nagpur
house_type_1 BHK Flat for Rent in Abhimaan Homes, Shirgaon, Pune
house_type_1 BHK Flat for Rent in Abhyudaya Nagar, Mumbai
house_type_1 BHK Flat for Rent in Acme Aakansha I, Goregaon Mulund Link Road, Mumbai
house_type_1 BHK Flat for Rent in Acme Harmony Chs Ltd, Andheri East, Mumbai
house_type_1 BHK Flat for Rent in Adarsh Nagar Lohegaon, Pune
house_type_1 BHK Flat for Rent in Adarsh Nagar, New Delhi
house_type_1 BHK Flat for Rent in Adithansh Apartment, Kondhwa, Pune
house_type_1 BHK Flat for Rent in Aditya Shagun, Bavdhan, Pune
house_type_1 BHK Flat for Rent in Aga Nagar, Vadgaonsheri, Pune
house_type_1 BHK Flat for Rent in Aishwarya Aangan, Chakan, Pune
house_type_1 BHK Flat for Rent in Ajmera Greenfinity, Wadala East, Mumbai
house_type_1 BHK Flat for Rent in Ajmera society, Bhavani Peth, Kasb

In [26]:
prefixes = {
    "house_type": "house_type_",
    "locality": "locality_",
    "furnishing": "furnishing_",
    "city": "city_"
}

print("=" * 70)
print("ENCODED FEATURE COUNTS")
print("=" * 70)

for name, prefix in prefixes.items():
    cols = [
        c for c in df.columns
        if c.startswith(prefix)
    ]

    print(f"{name:15}: {len(cols)}")
    

ENCODED FEATURE COUNTS
house_type     : 5466
locality       : 0
furnishing     : 2
city           : 4


In [27]:
raw_original = pd.read_csv(
    "../data/raw/cities_magicbricks_rental_prices.csv"
)

print("=" * 70)
print("RAW HOUSE TYPE")
print("=" * 70)

print("Rows:", len(raw_original))

print("\nUnique house_type values:",
      raw_original["house_type"].nunique())

print("\nSample values:")

for value in raw_original["house_type"].dropna().sample(
    min(30, raw_original["house_type"].notna().sum()),
    random_state=42
):
    print(value)

RAW HOUSE TYPE
Rows: 7691

Unique house_type values: 5467

Sample values:
3 BHK Flat for Rent in Juhu, Mumbai
3 BHK Flat for Rent in Moti Rattan Apartments, Dhirpur Village, Model Town, New Delhi
3 BHK Flat for Rent in Panchsheel Enclave, New Delhi
3 BHK Flat for Rent in Bhoganhalli, Bangalore
4 BHK House for Rent in Laburnum Park, Magarpatta Pune
3 BHK Flat for Rent in Sunteck City Avenue 1, Goregaon West, Mumbai
2 BHK Flat for Rent in Kirari Suleman Nagar, New Delhi
1 BHK House for Rent in Jaitala Nagpur
1 BHK Flat for Rent in Khanna Apartment CHS, Govind Nagar Borivali West, Mumbai
3 BHK Flat for Rent in Rivali Park, Magathane, Mumbai
5 BHK House for Rent in Kasturi Nagar Bangalore
1 BHK House for Rent in Wadgaon Sheri Pune
3 BHK Flat for Rent in Atmananda Colony, Hebbal, Bangalore
1 BHK House for Rent in Dhanori Pune
2 BHK Flat for Rent in Lohegaon, Pune
3 BHK Flat for Rent in Astron, Kandivali East, Mumbai
1 BHK House for Rent in RMV Extension Stage 2nd RMV Bangalore
1 BHK House f

In [28]:
print("\nMost common:")
print(
    raw_original["house_type"]
    .value_counts()
    .head(30)
)


Most common:
house_type
3 BHK Flat for Rent in Whitefield, Bangalore                     28
3 BHK Flat for Rent in Hebbal, Bangalore                         24
3 BHK Flat for Rent in Sarjapur Road, Bangalore                  24
3 BHK Flat for Rent in Oberoi Sky City, Borivali East, Mumbai    20
1 BHK Flat for Rent in Hadapsar, Pune                            19
3 BHK Flat for Rent in Shiv Kailasa, Mihan, Nagpur               16
2 BHK Flat for Rent in Whitefield, Bangalore                     16
2 BHK Flat for Rent in Marathahalli, Bangalore                   12
2 BHK Flat for Rent in Sarjapur Road, Bangalore                  12
3 BHK Flat for Rent in Yelahanka, Bangalore                      12
1 BHK House for Rent in Hadapsar Pune                            11
2 BHK Flat for Rent in Hinjawadi, Pune                           11
1 BHK Flat for Rent in Whitefield, Bangalore                     11
2 BHK Flat for Rent in Chattarpur, New Delhi                     10
2 BHK Flat for Rent in 

In [29]:
import re

def extract_property_type(text):
    text = str(text).lower()

    if "flat" in text or "apartment" in text:
        return "Flat"
    elif "villa" in text:
        return "Villa"
    elif "house" in text:
        return "House"
    elif "penthouse" in text:
        return "Penthouse"
    elif "studio" in text:
        return "Studio"
    elif "builder floor" in text:
        return "Builder Floor"
    else:
        return "Other"


raw_original["property_type"] = (
    raw_original["house_type"]
    .apply(extract_property_type)
)

print("=" * 70)
print("PROPERTY TYPE DISTRIBUTION")
print("=" * 70)

print(raw_original["property_type"].value_counts())

print("\nPercentages:")
print(
    (
        raw_original["property_type"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

PROPERTY TYPE DISTRIBUTION
property_type
Flat     5901
House    1539
Villa     251
Name: count, dtype: int64

Percentages:
property_type
Flat     76.73
House    20.01
Villa     3.26
Name: proportion, dtype: float64


In [30]:
property_keys = [
    "locality",
    "area",
    "beds",
    "bathrooms",
    "balconies",
    "rent"
]

property_type_map = (
    raw_original[
        property_keys + ["property_type"]
    ]
    .drop_duplicates(subset=property_keys)
)

print("Mapping rows:", len(property_type_map))

Mapping rows: 7663


In [31]:
df = df.merge(
    property_type_map,
    on=property_keys,
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("PROPERTY TYPE MERGE")
print("=" * 70)

print("Shape:", df.shape)

print("\nMissing property type:")
print(df["property_type"].isna().sum())

print("\nDistribution:")
print(df["property_type"].value_counts(dropna=False))

PROPERTY TYPE MERGE
Shape: (5286, 5492)

Missing property type:
0

Distribution:
property_type
Flat     4008
House    1104
Villa     174
Name: count, dtype: int64


In [32]:
house_type_cols = [
    col for col in df.columns
    if col.startswith("house_type_")
]

print("Removing old house_type columns:", len(house_type_cols))

df_clean_model = df.drop(
    columns=house_type_cols
).copy()

print("Before:", df.shape)
print("After :", df_clean_model.shape)

Removing old house_type columns: 5466
Before: (5286, 5492)
After : (5286, 26)


In [33]:
df_clean_model = pd.get_dummies(
    df_clean_model,
    columns=["property_type"],
    prefix="property_type",
    drop_first=False,
    dtype=int
)

property_cols = [
    col for col in df_clean_model.columns
    if col.startswith("property_type_")
]

print("Property type features:")
print(property_cols)

print("\nNew dataset shape:", df_clean_model.shape)

Property type features:
['property_type_Flat', 'property_type_House', 'property_type_Villa']

New dataset shape: (5286, 28)


In [34]:
print("=" * 70)
print("CLEAN MODEL DATASET")
print("=" * 70)

print("Shape:", df_clean_model.shape)

print("\nColumns:")
for col in df_clean_model.columns:
    print(col)

CLEAN MODEL DATASET
Shape: (5286, 28)

Columns:
locality
area
beds
bathrooms
balconies
area_rate
rent
city_Mumbai
city_Nagpur
city_New Delhi
city_Pune
furnishing_Semi-Furnished
furnishing_Unfurnished
area_per_bedroom
bath_per_bedroom
balcony_per_bedroom
city
latitude
longitude
hospital_km
school_km
mall_km
station_km
accessibility_score
accessibility_available
property_type_Flat
property_type_House
property_type_Villa


In [35]:
# ============================================================
# REMOVE UNUSED CITY DUMMIES
# ============================================================

unused_city_cols = [
    "city_Nagpur",
    "city_Pune"
]

df_clean_model = df_clean_model.drop(
    columns=[c for c in unused_city_cols if c in df_clean_model.columns]
)

print("Shape after removing unused city columns:",
      df_clean_model.shape)

print("\nCity dummy columns remaining:")
print([
    c for c in df_clean_model.columns
    if c.startswith("city_")
])

Shape after removing unused city columns: (5286, 26)

City dummy columns remaining:
['city_Mumbai', 'city_New Delhi']


In [36]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean_model,
    test_size=0.20,
    random_state=42
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (4228, 26)
Test : (1058, 26)


In [37]:
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

train_df = train_df.copy()
test_df = test_df.copy()

train_df["locality_market_rate"] = np.nan

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (fit_idx, val_idx) in enumerate(kf.split(train_df), 1):

    fit_part = train_df.iloc[fit_idx]
    val_part = train_df.iloc[val_idx]

    locality_rates = (
        fit_part
        .groupby(["city", "locality"])["area_rate"]
        .median()
    )

    city_rates = (
        fit_part
        .groupby("city")["area_rate"]
        .median()
    )

    keys = pd.MultiIndex.from_frame(
        val_part[["city", "locality"]]
    )

    mapped = locality_rates.reindex(keys).to_numpy()

    mapped = pd.Series(
        mapped,
        index=val_idx
    )

    fallback = (
        val_part["city"]
        .map(city_rates)
        .to_numpy()
    )

    mapped = mapped.fillna(
        pd.Series(
            fallback,
            index=val_idx
        )
    )

    train_df.iloc[
        val_idx,
        train_df.columns.get_loc("locality_market_rate")
    ] = mapped.to_numpy()

    print(f"Fold {fold} complete")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [38]:
final_locality_rates = (
    train_df
    .groupby(["city", "locality"])["area_rate"]
    .median()
)

final_city_rates = (
    train_df
    .groupby("city")["area_rate"]
    .median()
)

test_keys = pd.MultiIndex.from_frame(
    test_df[["city", "locality"]]
)

test_df["locality_market_rate"] = (
    final_locality_rates
    .reindex(test_keys)
    .to_numpy()
)

test_df["locality_market_rate"] = (
    test_df["locality_market_rate"]
    .fillna(
        test_df["city"].map(final_city_rates)
    )
)

print("Train market-rate missing:",
      train_df["locality_market_rate"].isna().sum())

print("Test market-rate missing:",
      test_df["locality_market_rate"].isna().sum())

Train market-rate missing: 0
Test market-rate missing: 0


In [39]:
exclude_cols = [
    "rent",
    "area_rate",      # target leakage
    "locality",       # raw string
    "city",           # raw string
    "latitude",       # raw/incomplete coordinates
    "longitude"
]

X_train_clean = train_df.drop(columns=exclude_cols)
X_test_clean  = test_df.drop(columns=exclude_cols)

y_train = train_df["rent"].copy()
y_test  = test_df["rent"].copy()

print("=" * 70)
print("CLEAN FEATURE MATRIX")
print("=" * 70)

print("X_train:", X_train_clean.shape)
print("X_test :", X_test_clean.shape)

print("\nMissing train:", X_train_clean.isna().sum().sum())
print("Missing test :", X_test_clean.isna().sum().sum())

print("\nFeatures:")
for col in X_train_clean.columns:
    print("•", col)

CLEAN FEATURE MATRIX
X_train: (4228, 21)
X_test : (1058, 21)

Missing train: 0
Missing test : 0

Features:
• area
• beds
• bathrooms
• balconies
• city_Mumbai
• city_New Delhi
• furnishing_Semi-Furnished
• furnishing_Unfurnished
• area_per_bedroom
• bath_per_bedroom
• balcony_per_bedroom
• hospital_km
• school_km
• mall_km
• station_km
• accessibility_score
• accessibility_available
• property_type_Flat
• property_type_House
• property_type_Villa
• locality_market_rate


In [40]:
from xgboost import XGBRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

clean_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.025,
    max_depth=5,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.1,
    reg_lambda=1.5,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training clean model...")

clean_model.fit(
    X_train_clean,
    y_train
)

pred_clean = clean_model.predict(X_test_clean)

print("✅ Training complete")

Training clean model...


✅ Training complete


In [41]:
r2_clean = r2_score(y_test, pred_clean)
mae_clean = mean_absolute_error(y_test, pred_clean)
rmse_clean = np.sqrt(
    mean_squared_error(y_test, pred_clean)
)

print("=" * 70)
print("CLEAN XGBOOST RESULTS")
print("=" * 70)

print(f"R²   : {r2_clean:.4f}")
print(f"MAE  : ₹{mae_clean:,.2f}")
print(f"RMSE : ₹{rmse_clean:,.2f}")

CLEAN XGBOOST RESULTS
R²   : 0.6913
MAE  : ₹21,663.62
RMSE : ₹53,384.30


In [42]:
# ============================================================
# OOF LOCALITY + BEDROOM MARKET RATE
# ============================================================

train_df["locality_bed_market_rate"] = np.nan

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (fit_idx, val_idx) in enumerate(kf.split(train_df), 1):

    fit_part = train_df.iloc[fit_idx]
    val_part = train_df.iloc[val_idx]

    local_bed_rates = (
        fit_part
        .groupby(["city", "locality", "beds"])["area_rate"]
        .median()
    )

    locality_rates = (
        fit_part
        .groupby(["city", "locality"])["area_rate"]
        .median()
    )

    city_rates = (
        fit_part
        .groupby("city")["area_rate"]
        .median()
    )

    keys = pd.MultiIndex.from_frame(
        val_part[["city", "locality", "beds"]]
    )

    # IMPORTANT: writable copy
    values = (
        local_bed_rates
        .reindex(keys)
        .to_numpy()
        .copy()
    )

    # Locality fallback
    missing = pd.isna(values)

    if missing.any():

        locality_keys = pd.MultiIndex.from_frame(
            val_part.loc[
                missing,
                ["city", "locality"]
            ]
        )

        fallback_values = (
            locality_rates
            .reindex(locality_keys)
            .to_numpy()
        )

        values[missing] = fallback_values

    # City fallback
    missing = pd.isna(values)

    if missing.any():

        city_fallback = (
            val_part.loc[missing, "city"]
            .map(city_rates)
            .to_numpy()
        )

        values[missing] = city_fallback

    train_df.iloc[
        val_idx,
        train_df.columns.get_loc(
            "locality_bed_market_rate"
        )
    ] = values

    print(f"Fold {fold} complete")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [43]:
final_local_bed_rates = (
    train_df
    .groupby(["city", "locality", "beds"])["area_rate"]
    .median()
)

final_locality_rates = (
    train_df
    .groupby(["city", "locality"])["area_rate"]
    .median()
)

final_city_rates = (
    train_df
    .groupby("city")["area_rate"]
    .median()
)

test_keys = pd.MultiIndex.from_frame(
    test_df[["city", "locality", "beds"]]
)

test_df["locality_bed_market_rate"] = (
    final_local_bed_rates
    .reindex(test_keys)
    .to_numpy()
)

# Locality fallback
missing = test_df["locality_bed_market_rate"].isna()

if missing.any():

    locality_keys = pd.MultiIndex.from_frame(
        test_df.loc[missing, ["city", "locality"]]
    )

    test_df.loc[
        missing,
        "locality_bed_market_rate"
    ] = (
        final_locality_rates
        .reindex(locality_keys)
        .to_numpy()
    )

# City fallback
missing = test_df["locality_bed_market_rate"].isna()

test_df.loc[
    missing,
    "locality_bed_market_rate"
] = (
    test_df.loc[missing, "city"]
    .map(final_city_rates)
)

print("=" * 70)
print("LOCALITY + BED MARKET RATE")
print("=" * 70)

print(
    "Train missing:",
    train_df["locality_bed_market_rate"].isna().sum()
)

print(
    "Test missing:",
    test_df["locality_bed_market_rate"].isna().sum()
)

print("\nTrain summary:")
print(train_df["locality_bed_market_rate"].describe())

LOCALITY + BED MARKET RATE
Train missing: 0
Test missing: 0

Train summary:
count    4228.000000
mean       51.550418
std        38.458981
min         4.000000
25%        28.000000
50%        34.250000
75%        74.000000
max       298.000000
Name: locality_bed_market_rate, dtype: float64


In [44]:
exclude_cols = [
    "rent",
    "area_rate",
    "locality",
    "city",
    "latitude",
    "longitude"
]

X_train_v2 = train_df.drop(columns=exclude_cols)
X_test_v2  = test_df.drop(columns=exclude_cols)

y_train = train_df["rent"].copy()
y_test  = test_df["rent"].copy()

print("X_train:", X_train_v2.shape)
print("X_test :", X_test_v2.shape)

print("Missing train:", X_train_v2.isna().sum().sum())
print("Missing test :", X_test_v2.isna().sum().sum())

print("\nMarket features:")
print("locality_market_rate" in X_train_v2.columns)
print("locality_bed_market_rate" in X_train_v2.columns)

X_train: (4228, 22)
X_test : (1058, 22)
Missing train: 0
Missing test : 0

Market features:
True
True


In [45]:
model_v2 = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.025,
    max_depth=5,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.1,
    reg_lambda=1.5,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training V2 model...")

model_v2.fit(
    X_train_v2,
    y_train
)

pred_v2 = model_v2.predict(X_test_v2)

print("✅ Training complete")

Training V2 model...
✅ Training complete


In [46]:
r2_v2 = r2_score(y_test, pred_v2)
mae_v2 = mean_absolute_error(y_test, pred_v2)
rmse_v2 = np.sqrt(
    mean_squared_error(y_test, pred_v2)
)

print("=" * 70)
print("XGBOOST V2 RESULTS")
print("=" * 70)

print(f"R²   : {r2_v2:.4f}")
print(f"MAE  : ₹{mae_v2:,.2f}")
print(f"RMSE : ₹{rmse_v2:,.2f}")

print("\nCHANGE VS CLEAN V1")
print("=" * 70)

print(f"R²   : {r2_clean:.4f} → {r2_v2:.4f}")
print(f"MAE  : ₹{mae_clean:,.2f} → ₹{mae_v2:,.2f}")
print(f"RMSE : ₹{rmse_clean:,.2f} → ₹{rmse_v2:,.2f}")

XGBOOST V2 RESULTS
R²   : 0.7028
MAE  : ₹21,755.87
RMSE : ₹52,386.47

CHANGE VS CLEAN V1
R²   : 0.6913 → 0.7028
MAE  : ₹21,663.62 → ₹21,755.87
RMSE : ₹53,384.30 → ₹52,386.47


In [47]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

base_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [500, 800, 1200, 1600],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "max_depth": [3, 4, 5, 6, 7],
    "min_child_weight": [1, 2, 4, 6],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.05, 0.1, 0.5, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0, 5.0]
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Starting hyperparameter search...")

search.fit(
    X_train_v2,
    y_train
)

print("\n✅ SEARCH COMPLETE")

print("\nBest parameters:")
print(search.best_params_)

print("\nBest CV RMSE:")
print(f"₹{-search.best_score_:,.2f}")

Starting hyperparameter search...
Fitting 3 folds for each of 25 candidates, totalling 75 fits

✅ SEARCH COMPLETE

Best parameters:
{'subsample': 1.0, 'reg_lambda': 0.5, 'reg_alpha': 0.05, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.7}

Best CV RMSE:
₹49,413.49


In [48]:
best_model = search.best_estimator_

pred_tuned = best_model.predict(X_test_v2)

r2_tuned = r2_score(y_test, pred_tuned)
mae_tuned = mean_absolute_error(y_test, pred_tuned)
rmse_tuned = np.sqrt(
    mean_squared_error(y_test, pred_tuned)
)

print("=" * 70)
print("TUNED XGBOOST RESULTS")
print("=" * 70)

print(f"R²   : {r2_tuned:.4f}")
print(f"MAE  : ₹{mae_tuned:,.2f}")
print(f"RMSE : ₹{rmse_tuned:,.2f}")

print("\nV2 → TUNED")
print("=" * 70)

print(f"R²   : {r2_v2:.4f} → {r2_tuned:.4f}")
print(f"MAE  : ₹{mae_v2:,.2f} → ₹{mae_tuned:,.2f}")
print(f"RMSE : ₹{rmse_v2:,.2f} → ₹{rmse_tuned:,.2f}")

TUNED XGBOOST RESULTS
R²   : 0.7125
MAE  : ₹21,857.27
RMSE : ₹51,526.85

V2 → TUNED
R²   : 0.7028 → 0.7125
MAE  : ₹21,755.87 → ₹21,857.27
RMSE : ₹52,386.47 → ₹51,526.85


In [49]:
evaluation = test_df[
    ["city", "locality", "rent"]
].copy()

evaluation["predicted_rent"] = pred_tuned

evaluation["absolute_error"] = (
    evaluation["rent"] -
    evaluation["predicted_rent"]
).abs()

evaluation["rent_band"] = pd.cut(
    evaluation["rent"],
    bins=[0, 20000, 50000, 100000, np.inf],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ]
)

print("=" * 70)
print("PERFORMANCE BY CITY")
print("=" * 70)

for city in evaluation["city"].unique():

    temp = evaluation[
        evaluation["city"] == city
    ]

    city_r2 = r2_score(
        temp["rent"],
        temp["predicted_rent"]
    )

    city_mae = mean_absolute_error(
        temp["rent"],
        temp["predicted_rent"]
    )

    city_rmse = np.sqrt(
        mean_squared_error(
            temp["rent"],
            temp["predicted_rent"]
        )
    )

    print(f"\n{city}")
    print(f"Properties : {len(temp)}")
    print(f"R²         : {city_r2:.4f}")
    print(f"MAE        : ₹{city_mae:,.2f}")
    print(f"RMSE       : ₹{city_rmse:,.2f}")

PERFORMANCE BY CITY

Bangalore
Properties : 352
R²         : 0.5538
MAE        : ₹22,342.64
RMSE       : ₹45,630.85

Mumbai
Properties : 341
R²         : 0.8323
MAE        : ₹26,519.54
RMSE       : ₹45,969.82

New Delhi
Properties : 365
R²         : 0.5947
MAE        : ₹17,033.49
RMSE       : ₹60,939.49


In [50]:
print("\n" + "=" * 70)
print("PERFORMANCE BY RENT BAND")
print("=" * 70)

band_results = (
    evaluation
    .groupby("rent_band", observed=True)
    .agg(
        properties=("rent", "size"),
        mean_actual=("rent", "mean"),
        mae=("absolute_error", "mean"),
        median_error=("absolute_error", "median")
    )
)

print(
    band_results.round(2)
)


PERFORMANCE BY RENT BAND
           properties  mean_actual       mae  median_error
rent_band                                                 
<₹20k             332     12764.76   8017.13       4728.95
₹20k–₹50k         349     34412.03  12263.99       8275.58
₹50k–₹1L          219     73442.92  20138.03      13881.34
>₹1L              158    230759.49  74512.31      49715.77


In [51]:
print("\n" + "=" * 70)
print("WORST 20 PREDICTIONS")
print("=" * 70)

print(
    evaluation[
        [
            "city",
            "locality",
            "rent",
            "predicted_rent",
            "absolute_error"
        ]
    ]
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(20)
    .round(2)
    .to_string(index=False)
)


WORST 20 PREDICTIONS
     city                  locality      rent  predicted_rent  absolute_error
New Delhi                  Sat Bari 1000000.0   164803.078125       835196.92
New Delhi Greater Kailash Enclave 1  850000.0   348428.812500       501571.19
New Delhi            Shanti Niketan  950000.0   516774.093750       433225.91
Bangalore         Hunasamaranahalli  550000.0   171996.906250       378003.09
   Mumbai           Napean Sea Road  700000.0   369117.531250       330882.50
Bangalore          Cambridge Layout  300000.0    40952.519531       259047.48
Bangalore                 Kempapura  350000.0    93075.578125       256924.42
New Delhi        Panchsheel Enclave  350000.0   108258.101562       241741.90
Bangalore              Lavelle Road  350000.0   121680.703125       228319.30
   Mumbai                Prabhadevi  250000.0   467050.625000       217050.62
   Mumbai                Prabhadevi  600000.0   797385.250000       197385.31
   Mumbai                Prabhadevi  75000

In [52]:
worst_indices = (
    evaluation
    .sort_values("absolute_error", ascending=False)
    .head(20)
    .index
)

inspect_cols = [
    "city",
    "locality",
    "rent",
    "area",
    "beds",
    "bathrooms",
    "balconies",
    "area_rate",
    "locality_market_rate",
    "locality_bed_market_rate",
    "accessibility_score"
]

worst_details = test_df.loc[
    worst_indices,
    inspect_cols
].copy()

worst_details["predicted_rent"] = (
    evaluation.loc[worst_indices, "predicted_rent"]
)

worst_details["absolute_error"] = (
    evaluation.loc[worst_indices, "absolute_error"]
)

print(
    worst_details
    .sort_values("absolute_error", ascending=False)
    .round(2)
    .to_string(index=False)
)

     city                  locality      rent   area  beds  bathrooms  balconies  area_rate  locality_market_rate  locality_bed_market_rate  accessibility_score  predicted_rent  absolute_error
New Delhi                  Sat Bari 1000000.0 4500.0     5          5          5     221.00                  28.0                      28.0                 8.88   164803.078125       835196.92
New Delhi Greater Kailash Enclave 1  850000.0 4050.0    10          0          0     210.00                  50.0                      50.0                 9.07   348428.812500       501571.19
New Delhi            Shanti Niketan  950000.0 9000.0     4          4          4     105.56                 138.0                     138.0                 8.72   516774.093750       433225.91
Bangalore         Hunasamaranahalli  550000.0 6700.0     5          7          5      82.00                  23.0                      23.0                 5.33   171996.906250       378003.09
   Mumbai           Napean Sea Road

In [53]:
print("=" * 70)
print("PERFORMANCE AT DIFFERENT RENT LEVELS")
print("=" * 70)

thresholds = [
    100000,
    200000,
    300000,
    500000,
    1000000
]

for limit in thresholds:

    mask = y_test <= limit

    r2_temp = r2_score(
        y_test[mask],
        pred_tuned[mask]
    )

    mae_temp = mean_absolute_error(
        y_test[mask],
        pred_tuned[mask]
    )

    rmse_temp = np.sqrt(
        mean_squared_error(
            y_test[mask],
            pred_tuned[mask]
        )
    )

    print(
        f"Rent ≤ ₹{limit:,} | "
        f"N={mask.sum()} | "
        f"R²={r2_temp:.4f} | "
        f"MAE=₹{mae_temp:,.0f} | "
        f"RMSE=₹{rmse_temp:,.0f}"
    )

PERFORMANCE AT DIFFERENT RENT LEVELS
Rent ≤ ₹100,000 | N=900 | R²=0.3604 | MAE=₹12,613 | RMSE=₹20,105
Rent ≤ ₹200,000 | N=1002 | R²=0.6049 | MAE=₹15,502 | RMSE=₹26,103
Rent ≤ ₹300,000 | N=1028 | R²=0.6631 | MAE=₹17,347 | RMSE=₹30,788
Rent ≤ ₹500,000 | N=1047 | R²=0.7249 | MAE=₹19,150 | RMSE=₹35,687
Rent ≤ ₹1,000,000 | N=1058 | R²=0.7125 | MAE=₹21,857 | RMSE=₹51,527


In [54]:
# ============================================================
# RATE-PREDICTION MODEL
# ============================================================

# IMPORTANT:
# area_rate is TARGET here, not an input feature.

y_rate_train = train_df["area_rate"].copy()
y_rate_test = test_df["area_rate"].copy()

rate_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.025,
    max_depth=5,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.1,
    reg_lambda=1.5,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

rate_model.fit(
    X_train_v2,
    y_rate_train
)

predicted_rate = rate_model.predict(X_test_v2)

# Prevent impossible negative rental rates
predicted_rate = np.maximum(predicted_rate, 0)

# Convert predicted ₹/sqft back into monthly rent
pred_rent_from_rate = (
    predicted_rate *
    test_df["area"].to_numpy()
)

In [55]:
# ============================================================
# RATE MODEL EVALUATION
# ============================================================

rate_r2 = r2_score(
    y_rate_test,
    predicted_rate
)

rate_mae = mean_absolute_error(
    y_rate_test,
    predicted_rate
)

rent_r2_rate_model = r2_score(
    y_test,
    pred_rent_from_rate
)

rent_mae_rate_model = mean_absolute_error(
    y_test,
    pred_rent_from_rate
)

rent_rmse_rate_model = np.sqrt(
    mean_squared_error(
        y_test,
        pred_rent_from_rate
    )
)

print("=" * 70)
print("RATE MODEL")
print("=" * 70)

print(f"Area-rate R²  : {rate_r2:.4f}")
print(f"Area-rate MAE : ₹{rate_mae:,.2f}/sqft")

print("\n" + "=" * 70)
print("RENT FROM PREDICTED RATE")
print("=" * 70)

print(f"R²   : {rent_r2_rate_model:.4f}")
print(f"MAE  : ₹{rent_mae_rate_model:,.2f}")
print(f"RMSE : ₹{rent_rmse_rate_model:,.2f}")

print("\nCurrent direct-rent benchmark:")
print("R²   : 0.7125")
print("MAE  : ₹21,857")
print("RMSE : ₹51,527")

RATE MODEL
Area-rate R²  : 0.6690
Area-rate MAE : ₹16.38/sqft

RENT FROM PREDICTED RATE
R²   : 0.7160
MAE  : ₹20,842.60
RMSE : ₹51,204.70

Current direct-rent benchmark:
R²   : 0.7125
MAE  : ₹21,857
RMSE : ₹51,527


In [56]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

rate_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

rate_params = {
    "n_estimators": [500, 800, 1200, 1600],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "max_depth": [3, 4, 5, 6, 7],
    "min_child_weight": [1, 2, 4, 6],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.05, 0.1, 0.5, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0, 5.0]
}

rate_search = RandomizedSearchCV(
    estimator=rate_base,
    param_distributions=rate_params,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

print("Tuning area-rate model...")

rate_search.fit(
    X_train_v2,
    y_rate_train
)

print("\n✅ RATE MODEL TUNING COMPLETE")

print("\nBest parameters:")
print(rate_search.best_params_)

print("\nBest CV area-rate RMSE:")
print(f"{-rate_search.best_score_:.2f}")

Tuning area-rate model...
Fitting 3 folds for each of 25 candidates, totalling 75 fits

✅ RATE MODEL TUNING COMPLETE

Best parameters:
{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 1200, 'min_child_weight': 6, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}

Best CV area-rate RMSE:
24.04


In [57]:
best_rate_model = rate_search.best_estimator_

predicted_rate_tuned = best_rate_model.predict(
    X_test_v2
)

predicted_rate_tuned = np.maximum(
    predicted_rate_tuned,
    0
)

pred_rent_tuned_rate = (
    predicted_rate_tuned *
    test_df["area"].to_numpy()
)

rate_r2_tuned = r2_score(
    y_rate_test,
    predicted_rate_tuned
)

rent_r2_tuned_rate = r2_score(
    y_test,
    pred_rent_tuned_rate
)

rent_mae_tuned_rate = mean_absolute_error(
    y_test,
    pred_rent_tuned_rate
)

rent_rmse_tuned_rate = np.sqrt(
    mean_squared_error(
        y_test,
        pred_rent_tuned_rate
    )
)

print("=" * 70)
print("TUNED RATE → RENT RESULTS")
print("=" * 70)

print(f"Area-rate R² : {rate_r2_tuned:.4f}")

print("\nFINAL RENT PERFORMANCE")
print(f"R²   : {rent_r2_tuned_rate:.4f}")
print(f"MAE  : ₹{rent_mae_tuned_rate:,.2f}")
print(f"RMSE : ₹{rent_rmse_tuned_rate:,.2f}")

print("\nPrevious rate-model benchmark:")
print("R²   : 0.7160")
print("MAE  : ₹20,842.60")
print("RMSE : ₹51,204.70")

TUNED RATE → RENT RESULTS
Area-rate R² : 0.6714

FINAL RENT PERFORMANCE
R²   : 0.7155
MAE  : ₹20,661.70
RMSE : ₹51,254.33

Previous rate-model benchmark:
R²   : 0.7160
MAE  : ₹20,842.60
RMSE : ₹51,204.70


In [58]:
comparison = pd.DataFrame({
    "actual": y_test.values,
    "untuned": pred_rent_from_rate,
    "tuned": pred_rent_tuned_rate
})

comparison["rent_band"] = pd.cut(
    comparison["actual"],
    bins=[0, 20000, 50000, 100000, np.inf],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ]
)

comparison["untuned_error"] = (
    comparison["actual"] - comparison["untuned"]
).abs()

comparison["tuned_error"] = (
    comparison["actual"] - comparison["tuned"]
).abs()

band_comparison = comparison.groupby(
    "rent_band",
    observed=True
).agg(
    count=("actual", "size"),
    untuned_MAE=("untuned_error", "mean"),
    tuned_MAE=("tuned_error", "mean"),
    untuned_median=("untuned_error", "median"),
    tuned_median=("tuned_error", "median")
)

band_comparison["MAE_improvement"] = (
    band_comparison["untuned_MAE"] -
    band_comparison["tuned_MAE"]
)

print("=" * 80)
print("UNTUNED VS TUNED RATE MODEL")
print("=" * 80)

print(band_comparison.round(2))

UNTUNED VS TUNED RATE MODEL
           count  untuned_MAE  tuned_MAE  untuned_median  tuned_median  \
rent_band                                                                
<₹20k        332      6205.95    6322.24         3797.16       4194.54   
₹20k–₹50k    349     11683.57   11237.22         7547.80       7223.62   
₹50k–₹1L     219     20701.40   20470.63        13324.52      14526.18   
>₹1L         158     72024.78   71874.94        43789.78      41528.19   

           MAE_improvement  
rent_band                   
<₹20k              -116.29  
₹20k–₹50k           446.36  
₹50k–₹1L            230.78  
>₹1L                149.84  


In [59]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

# Start from our clean 28-column dataset
production_df = df_clean_model.copy().reset_index(drop=True)

# Remove any old market-rate features if present
production_df = production_df.drop(
    columns=[
        "locality_market_rate",
        "locality_bed_market_rate"
    ],
    errors="ignore"
)

production_df["locality_market_rate"] = np.nan
production_df["locality_bed_market_rate"] = np.nan

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (fit_idx, val_idx) in enumerate(kf.split(production_df), 1):

    fit_part = production_df.iloc[fit_idx]
    val_part = production_df.iloc[val_idx]

    # --------------------------------------------------------
    # LOCALITY MARKET RATE
    # --------------------------------------------------------

    locality_rates = (
        fit_part
        .groupby(["city", "locality"])["area_rate"]
        .median()
    )

    city_rates = (
        fit_part
        .groupby("city")["area_rate"]
        .median()
    )

    locality_keys = pd.MultiIndex.from_frame(
        val_part[["city", "locality"]]
    )

    values = (
        locality_rates
        .reindex(locality_keys)
        .to_numpy()
        .copy()
    )

    missing = pd.isna(values)

    if missing.any():
        values[missing] = (
            val_part.loc[missing, "city"]
            .map(city_rates)
            .to_numpy()
        )

    production_df.loc[
        val_idx,
        "locality_market_rate"
    ] = values

    # --------------------------------------------------------
    # LOCALITY + BED MARKET RATE
    # --------------------------------------------------------

    local_bed_rates = (
        fit_part
        .groupby(["city", "locality", "beds"])["area_rate"]
        .median()
    )

    bed_keys = pd.MultiIndex.from_frame(
        val_part[["city", "locality", "beds"]]
    )

    bed_values = (
        local_bed_rates
        .reindex(bed_keys)
        .to_numpy()
        .copy()
    )

    # fallback → locality median
    missing = pd.isna(bed_values)

    if missing.any():

        fallback_keys = pd.MultiIndex.from_frame(
            val_part.loc[
                missing,
                ["city", "locality"]
            ]
        )

        bed_values[missing] = (
            locality_rates
            .reindex(fallback_keys)
            .to_numpy()
        )

    # fallback → city median
    missing = pd.isna(bed_values)

    if missing.any():

        bed_values[missing] = (
            val_part.loc[missing, "city"]
            .map(city_rates)
            .to_numpy()
        )

    production_df.loc[
        val_idx,
        "locality_bed_market_rate"
    ] = bed_values

    print(f"✅ Fold {fold} complete")


print("\n" + "=" * 70)
print("PRODUCTION FEATURE CHECK")
print("=" * 70)

print(
    "Locality market-rate missing:",
    production_df["locality_market_rate"].isna().sum()
)

print(
    "Locality+bed market-rate missing:",
    production_df["locality_bed_market_rate"].isna().sum()
)

print("Shape:", production_df.shape)

✅ Fold 1 complete
✅ Fold 2 complete
✅ Fold 3 complete
✅ Fold 4 complete
✅ Fold 5 complete

PRODUCTION FEATURE CHECK
Locality market-rate missing: 0
Locality+bed market-rate missing: 0
Shape: (5286, 28)


In [60]:
exclude_cols = [
    "rent",
    "area_rate",
    "locality",
    "city",
    "latitude",
    "longitude"
]

X_production = production_df.drop(
    columns=exclude_cols
)

y_production_rate = production_df[
    "area_rate"
].copy()

print("=" * 70)
print("FINAL PRODUCTION MATRIX")
print("=" * 70)

print("X:", X_production.shape)
print("y:", y_production_rate.shape)

print(
    "Missing X:",
    X_production.isna().sum().sum()
)

print("\nFeatures:")

for feature in X_production.columns:
    print("•", feature)

FINAL PRODUCTION MATRIX
X: (5286, 22)
y: (5286,)
Missing X: 0

Features:
• area
• beds
• bathrooms
• balconies
• city_Mumbai
• city_New Delhi
• furnishing_Semi-Furnished
• furnishing_Unfurnished
• area_per_bedroom
• bath_per_bedroom
• balcony_per_bedroom
• hospital_km
• school_km
• mall_km
• station_km
• accessibility_score
• accessibility_available
• property_type_Flat
• property_type_House
• property_type_Villa
• locality_market_rate
• locality_bed_market_rate


In [61]:
from xgboost import XGBRegressor

final_params = rate_search.best_params_.copy()

final_rate_model = XGBRegressor(
    **final_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training FINAL production model...")

final_rate_model.fit(
    X_production,
    y_production_rate
)

print("✅ FINAL MODEL TRAINED")
print("Training rows:", len(X_production))
print("Features:", X_production.shape[1])

Training FINAL production model...
✅ FINAL MODEL TRAINED
Training rows: 5286
Features: 22


In [62]:
import os
import pickle
import json
import pandas as pd
from datetime import datetime

# ============================================================
# SAVE FINAL TRUEESTATE PRODUCTION ARTIFACTS
# ============================================================

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. SAVE FINAL XGBOOST RATE MODEL
# ------------------------------------------------------------

with open(
    f"{MODEL_DIR}/trueestate_rate_model.pkl",
    "wb"
) as f:
    pickle.dump(final_rate_model, f)


# ------------------------------------------------------------
# 2. SAVE EXACT FEATURE ORDER
# ------------------------------------------------------------

feature_columns = X_production.columns.tolist()

with open(
    f"{MODEL_DIR}/trueestate_feature_columns.pkl",
    "wb"
) as f:
    pickle.dump(feature_columns, f)


# ------------------------------------------------------------
# 3. BUILD PRODUCTION MARKET-RATE LOOKUPS
#    These use ALL available properties.
# ------------------------------------------------------------

locality_rate_lookup = (
    production_df
    .groupby(["city", "locality"])["area_rate"]
    .median()
    .reset_index()
    .rename(
        columns={
            "area_rate": "locality_market_rate"
        }
    )
)

locality_bed_rate_lookup = (
    production_df
    .groupby(
        ["city", "locality", "beds"]
    )["area_rate"]
    .median()
    .reset_index()
    .rename(
        columns={
            "area_rate": "locality_bed_market_rate"
        }
    )
)

city_rate_lookup = (
    production_df
    .groupby("city")["area_rate"]
    .median()
    .reset_index()
    .rename(
        columns={
            "area_rate": "city_market_rate"
        }
    )
)


# ------------------------------------------------------------
# 4. SAVE MARKET LOOKUPS
# ------------------------------------------------------------

locality_rate_lookup.to_csv(
    f"{MODEL_DIR}/locality_market_rates.csv",
    index=False
)

locality_bed_rate_lookup.to_csv(
    f"{MODEL_DIR}/locality_bed_market_rates.csv",
    index=False
)

city_rate_lookup.to_csv(
    f"{MODEL_DIR}/city_market_rates.csv",
    index=False
)


# ------------------------------------------------------------
# 5. ACCESSIBILITY LOOKUP
# ------------------------------------------------------------

accessibility_cols = [
    "city",
    "locality",
    "hospital_km",
    "school_km",
    "mall_km",
    "station_km",
    "accessibility_score",
    "accessibility_available"
]

accessibility_lookup = (
    production_df[
        accessibility_cols
    ]
    .drop_duplicates(
        subset=["city", "locality"]
    )
    .reset_index(drop=True)
)

accessibility_lookup.to_csv(
    f"{MODEL_DIR}/accessibility_lookup.csv",
    index=False
)


# ------------------------------------------------------------
# 6. SAVE MODEL METADATA
# ------------------------------------------------------------

metadata = {

    "model_name":
        "TrueEstate Rental Rate XGBoost",

    "model_type":
        "XGBRegressor",

    "prediction_target":
        "monthly_rental_rate_per_sqft",

    "rent_calculation":
        "predicted_area_rate * area",

    "training_rows":
        int(len(X_production)),

    "feature_count":
        int(X_production.shape[1]),

    "cities": [
        "Bangalore",
        "Mumbai",
        "New Delhi"
    ],

    "validation_metrics": {
        "area_rate_r2": 0.6714,
        "rent_r2": 0.7155,
        "rent_mae": 20661.70,
        "rent_rmse": 51254.33
    },

    "model_parameters":
        final_params,

    "created_at":
        datetime.now().isoformat()
}


with open(
    f"{MODEL_DIR}/trueestate_model_metadata.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 7. FINAL CHECK
# ------------------------------------------------------------

files = [
    "trueestate_rate_model.pkl",
    "trueestate_feature_columns.pkl",
    "locality_market_rates.csv",
    "locality_bed_market_rates.csv",
    "city_market_rates.csv",
    "accessibility_lookup.csv",
    "trueestate_model_metadata.json"
]

print("=" * 70)
print("TRUEESTATE PRODUCTION ARTIFACTS")
print("=" * 70)

for file in files:

    path = f"{MODEL_DIR}/{file}"

    if os.path.exists(path):

        size_kb = os.path.getsize(path) / 1024

        print(
            f"✅ {file:<40} "
            f"{size_kb:>8.2f} KB"
        )

    else:

        print(
            f"❌ {file}"
        )

print("\nFinal model:")
print("Input features :", len(feature_columns))
print("Training rows  :", len(X_production))

print("\n🎉 TRUEESTATE MODEL READY FOR DEPLOYMENT")

TRUEESTATE PRODUCTION ARTIFACTS
✅ trueestate_rate_model.pkl                 3739.91 KB
✅ trueestate_feature_columns.pkl               0.40 KB
✅ locality_market_rates.csv                   46.42 KB
✅ locality_bed_market_rates.csv               78.50 KB
✅ city_market_rates.csv                        0.07 KB
✅ accessibility_lookup.csv                   157.76 KB
✅ trueestate_model_metadata.json               0.80 KB

Final model:
Input features : 22
Training rows  : 5286

🎉 TRUEESTATE MODEL READY FOR DEPLOYMENT


In [63]:
import pickle
import pandas as pd
import numpy as np

MODEL_DIR = "../models"

# Load model
with open(f"{MODEL_DIR}/trueestate_rate_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Load feature order
with open(f"{MODEL_DIR}/trueestate_feature_columns.pkl", "rb") as f:
    loaded_features = pickle.load(f)

# Load lookups
locality_rates = pd.read_csv(
    f"{MODEL_DIR}/locality_market_rates.csv"
)

locality_bed_rates = pd.read_csv(
    f"{MODEL_DIR}/locality_bed_market_rates.csv"
)

city_rates = pd.read_csv(
    f"{MODEL_DIR}/city_market_rates.csv"
)

accessibility_lookup = pd.read_csv(
    f"{MODEL_DIR}/accessibility_lookup.csv"
)

print("✅ Model loaded")
print("Expected features:", len(loaded_features))
print("Accessibility rows:", len(accessibility_lookup))

✅ Model loaded
Expected features: 22
Accessibility rows: 1511


In [64]:
def predict_rent(
    city,
    locality,
    area,
    beds,
    bathrooms,
    balconies,
    furnishing,
    property_type
):
    # -----------------------------
    # Market-rate lookup
    # -----------------------------
    local_bed_match = locality_bed_rates[
        (locality_bed_rates["city"] == city) &
        (locality_bed_rates["locality"] == locality) &
        (locality_bed_rates["beds"] == beds)
    ]

    if not local_bed_match.empty:
        locality_bed_market_rate = local_bed_match.iloc[0][
            "locality_bed_market_rate"
        ]
    else:
        locality_match = locality_rates[
            (locality_rates["city"] == city) &
            (locality_rates["locality"] == locality)
        ]

        if not locality_match.empty:
            locality_bed_market_rate = locality_match.iloc[0][
                "locality_market_rate"
            ]
        else:
            locality_bed_market_rate = city_rates.loc[
                city_rates["city"] == city,
                "city_market_rate"
            ].iloc[0]

    locality_match = locality_rates[
        (locality_rates["city"] == city) &
        (locality_rates["locality"] == locality)
    ]

    if not locality_match.empty:
        locality_market_rate = locality_match.iloc[0][
            "locality_market_rate"
        ]
    else:
        locality_market_rate = city_rates.loc[
            city_rates["city"] == city,
            "city_market_rate"
        ].iloc[0]

    # -----------------------------
    # Accessibility lookup
    # -----------------------------
    access = accessibility_lookup[
        (accessibility_lookup["city"] == city) &
        (accessibility_lookup["locality"] == locality)
    ]

    if not access.empty:
        access = access.iloc[0]

        hospital_km = access["hospital_km"]
        school_km = access["school_km"]
        mall_km = access["mall_km"]
        station_km = access["station_km"]
        accessibility_score = access["accessibility_score"]
        accessibility_available = access["accessibility_available"]

    else:
        # city-level fallback
        city_access = accessibility_lookup[
            accessibility_lookup["city"] == city
        ]

        hospital_km = city_access["hospital_km"].median()
        school_km = city_access["school_km"].median()
        mall_km = city_access["mall_km"].median()
        station_km = city_access["station_km"].median()
        accessibility_score = city_access["accessibility_score"].median()
        accessibility_available = 0

    # -----------------------------
    # Build model input
    # -----------------------------
    row = {feature: 0 for feature in loaded_features}

    row["area"] = area
    row["beds"] = beds
    row["bathrooms"] = bathrooms
    row["balconies"] = balconies

    row["area_per_bedroom"] = area / beds if beds > 0 else area
    row["bath_per_bedroom"] = bathrooms / beds if beds > 0 else bathrooms
    row["balcony_per_bedroom"] = balconies / beds if beds > 0 else balconies

    row["locality_market_rate"] = locality_market_rate
    row["locality_bed_market_rate"] = locality_bed_market_rate

    row["hospital_km"] = hospital_km
    row["school_km"] = school_km
    row["mall_km"] = mall_km
    row["station_km"] = station_km
    row["accessibility_score"] = accessibility_score
    row["accessibility_available"] = accessibility_available

    # City dummies
    if city == "Mumbai":
        row["city_Mumbai"] = 1
    elif city == "New Delhi":
        row["city_New Delhi"] = 1

    # Furnishing dummies
    if furnishing == "Semi-Furnished":
        row["furnishing_Semi-Furnished"] = 1
    elif furnishing == "Unfurnished":
        row["furnishing_Unfurnished"] = 1

    # Property type
    property_col = f"property_type_{property_type}"

    if property_col in row:
        row[property_col] = 1

    X_user = pd.DataFrame(
        [row],
        columns=loaded_features
    )

    # predict ₹/sqft
    predicted_rate = float(
        loaded_model.predict(X_user)[0]
    )

    predicted_rate = max(predicted_rate, 0)

    predicted_rent = predicted_rate * area

    return {
        "predicted_rate_per_sqft": round(predicted_rate, 2),
        "predicted_monthly_rent": round(predicted_rent, 2),
        "accessibility_score": round(float(accessibility_score), 2),
        "locality_market_rate": round(float(locality_market_rate), 2)
    }

In [65]:
result = predict_rent(
    city="Bangalore",
    locality="Whitefield",
    area=1200,
    beds=2,
    bathrooms=2,
    balconies=1,
    furnishing="Semi-Furnished",
    property_type="Flat"
)

print(result)

{'predicted_rate_per_sqft': 38.05, 'predicted_monthly_rent': 45661.43, 'accessibility_score': 9.44, 'locality_market_rate': 39.5}


In [66]:
# ============================================================
# TRUEESTATE VALUATION ERROR DISTRIBUTION
# Used to build statistically grounded fair-rent ranges
# ============================================================

import pandas as pd
import numpy as np

valuation_errors = pd.DataFrame({
    "actual_rent": y_test.values,
    "predicted_rent": pred_rent_tuned_rate
})

# Signed residual:
# positive = model underpredicted
# negative = model overpredicted
valuation_errors["residual"] = (
    valuation_errors["actual_rent"]
    - valuation_errors["predicted_rent"]
)

valuation_errors["absolute_error"] = (
    valuation_errors["residual"].abs()
)

valuation_errors["absolute_percentage_error"] = (
    valuation_errors["absolute_error"]
    / valuation_errors["actual_rent"]
) * 100


# ------------------------------------------------------------
# RENT BANDS
# ------------------------------------------------------------

valuation_errors["rent_band"] = pd.cut(
    valuation_errors["actual_rent"],
    bins=[
        0,
        20000,
        50000,
        100000,
        np.inf
    ],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ],
    include_lowest=True
)


# ------------------------------------------------------------
# ABSOLUTE ERROR PERCENTILES
# ------------------------------------------------------------

error_percentiles = (
    valuation_errors
    .groupby(
        "rent_band",
        observed=True
    )["absolute_error"]
    .quantile([
        0.50,
        0.80,
        0.90,
        0.95
    ])
    .unstack()
)

error_percentiles.columns = [
    "P50_error",
    "P80_error",
    "P90_error",
    "P95_error"
]


# ------------------------------------------------------------
# PERCENTAGE ERROR PERCENTILES
# ------------------------------------------------------------

percentage_percentiles = (
    valuation_errors
    .groupby(
        "rent_band",
        observed=True
    )["absolute_percentage_error"]
    .quantile([
        0.50,
        0.80,
        0.90,
        0.95
    ])
    .unstack()
)

percentage_percentiles.columns = [
    "P50_pct",
    "P80_pct",
    "P90_pct",
    "P95_pct"
]


# ------------------------------------------------------------
# SAMPLE COUNTS
# ------------------------------------------------------------

counts = (
    valuation_errors
    .groupby(
        "rent_band",
        observed=True
    )
    .size()
    .rename("properties")
)


# ------------------------------------------------------------
# COMBINE RESULTS
# ------------------------------------------------------------

valuation_uncertainty = pd.concat(
    [
        counts,
        error_percentiles,
        percentage_percentiles
    ],
    axis=1
)


print("=" * 90)
print("TRUEESTATE VALUATION UNCERTAINTY")
print("=" * 90)

print(
    valuation_uncertainty.round(2)
)

TRUEESTATE VALUATION UNCERTAINTY
           properties  P50_error  P80_error  P90_error  P95_error  P50_pct  \
rent_band                                                                    
<₹20k             332    4194.54    8740.06   13065.91   19784.34    33.36   
₹20k–₹50k         349    7223.62   17404.93   24671.66   31677.21    22.77   
₹50k–₹1L          219   14526.18   30978.63   46533.14   64201.37    19.66   
>₹1L              158   41528.19  101246.21  166297.61  224073.45    20.62   

           P80_pct  P90_pct  P95_pct  
rent_band                             
<₹20k        88.25   121.68   196.59  
₹20k–₹50k    47.28    71.28    96.06  
₹50k–₹1L     44.34    64.61    82.52  
>₹1L         46.84    61.74    75.84  


In [67]:
# ============================================================
# TRUEESTATE PREDICTION-CONDITIONAL CALIBRATION
# ============================================================

calibration = pd.DataFrame({
    "actual_rent": y_test.values,
    "predicted_rent": pred_rent_tuned_rate
})

calibration["residual"] = (
    calibration["actual_rent"]
    - calibration["predicted_rent"]
)


# Create bands using PREDICTED rent
calibration["prediction_band"] = pd.cut(
    calibration["predicted_rent"],
    bins=[
        0,
        20000,
        50000,
        100000,
        np.inf
    ],
    labels=[
        "<₹20k",
        "₹20k–₹50k",
        "₹50k–₹1L",
        ">₹1L"
    ],
    include_lowest=True
)


# ------------------------------------------------------------
# SIGNED RESIDUAL QUANTILES
#
# 10th + 90th percentile gives an empirical central
# 80% prediction interval.
# ------------------------------------------------------------

residual_intervals = (
    calibration
    .groupby(
        "prediction_band",
        observed=True
    )["residual"]
    .quantile([
        0.10,
        0.50,
        0.90
    ])
    .unstack()
)

residual_intervals.columns = [
    "Q10_residual",
    "median_residual",
    "Q90_residual"
]


# Number of validation properties in each band
counts = (
    calibration
    .groupby(
        "prediction_band",
        observed=True
    )
    .size()
    .rename("properties")
)


calibration_summary = pd.concat(
    [
        counts,
        residual_intervals
    ],
    axis=1
)


print("=" * 90)
print("TRUEESTATE PREDICTION-CONDITIONAL CALIBRATION")
print("=" * 90)

print(
    calibration_summary.round(2)
)

TRUEESTATE PREDICTION-CONDITIONAL CALIBRATION
                 properties  Q10_residual  median_residual  Q90_residual
prediction_band                                                         
<₹20k                   272      -6956.94          -396.01       7739.73
₹20k–₹50k               356     -14637.02         -1962.29      14585.33
₹50k–₹1L                256     -29969.19         -5407.85      29411.62
>₹1L                    174     -93083.96         -8423.03      91786.42


In [1]:
# ============================================================
# RECOMMENDATION ENGINE DATA CHECK
# ============================================================

import pandas as pd


files = {
    "accessibility":
        "../models/accessibility_lookup.csv",

    "locality_rates":
        "../models/locality_market_rates.csv",

    "locality_bed_rates":
        "../models/locality_bed_market_rates.csv",

    "city_rates":
        "../models/city_market_rates.csv"
}


for name, path in files.items():

    df_check = pd.read_csv(path)

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print("Shape:", df_check.shape)

    print("\nColumns:")
    print(df_check.columns.tolist())

    print("\nSample:")
    print(df_check.head(3))


ACCESSIBILITY
Shape: (1511, 8)

Columns:
['city', 'locality', 'hospital_km', 'school_km', 'mall_km', 'station_km', 'accessibility_score', 'accessibility_available']

Sample:
     city       locality  hospital_km  school_km   mall_km  station_km  \
0  Mumbai  Goregaon East     0.353217   0.543976  0.425631    0.371553   
1  Mumbai          Powai     0.366617   0.145246  0.231953    2.316426   
2  Mumbai      Mira Road     0.000000   0.491363  0.332871    0.300411   

   accessibility_score  accessibility_available  
0                 9.31                        1  
1                 8.80                        1  
2                 9.59                        1  

LOCALITY_RATES
Shape: (1511, 3)

Columns:
['city', 'locality', 'locality_market_rate']

Sample:
        city              locality  locality_market_rate
0  Bangalore  1A Block Koramangala                  83.0
1  Bangalore        1st Block East                  33.0
2  Bangalore        A Narayanapura                  71.0

LO

In [2]:
print("LOCALITY RATES:")
print(pd.read_csv("../models/locality_market_rates.csv").columns.tolist())

print("\nLOCALITY BED RATES:")
print(pd.read_csv("../models/locality_bed_market_rates.csv").columns.tolist())

print("\nCITY RATES:")
print(pd.read_csv("../models/city_market_rates.csv").columns.tolist())

LOCALITY RATES:
['city', 'locality', 'locality_market_rate']

LOCALITY BED RATES:
['city', 'locality', 'beds', 'locality_bed_market_rate']

CITY RATES:
['city', 'city_market_rate']
